In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
   (2, "C002", "Mobile", None, "2024-01-02"),
   (3, "C003", "Tablet", "20000", "2024-01-03"),
   (4, "C004", "Laptop", "55000", "2024-01-04"),
   (5, "C005", "Headphones", None, "2024-01-05"),
   (6, "C006", "Camera", "30000", "2024-01-06"),
   (7, "C007", "Mobile", "18000", "2024-01-07"),
   (8, "C008", "Watch", "8000", "2024-01-07")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import col, when, to_date, udf, lit
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql import *

#  Cast Columns - Convert amount → integer

df = df.withColumn("amount", col("amount").cast(IntegerType()))

# Task 1: Handle NULLs - Replace NULL amount with 1000
df = df.fillna({"amount" : 1000})

df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
# Task 2: Cast Columns - Convert string type updated_date → to date format
df = df.withColumn("updated_date", to_date(col("updated_date")))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
# Add Derived Columns as bonus = amount * 0.1, category:20000 → High else → Low
final_df = df.withColumn("bonus", col("amount") * 0.1)

final_df = final_df.withColumn(
    "category",
    when(col("amount") >= 20000, "High").otherwise("Low")
)
final_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


In [0]:
# Create amount_bucket:
# < 10000 → Low
# 10000–30000 → Medium
# 30000 → High
def bucket_func(amount):
    if amount < 10000:
        return "Low"
    elif 10000 <= amount <= 30000:
        return "Medium"
    else:
        return "High"

bucket_udf = udf(bucket_func, StringType())

final_df = final_df.withColumn("amount_bucket", bucket_udf(col("amount")))
final_df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


In [0]:
# full load
output_path = "/Volumes/workspace/default/outputs"

df.write.mode("overwrite").parquet("/Volumes/workspace/default/outputs/load")

In [0]:
# incremental loading
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06"),
    (9, "C009", "Speaker", "15000", "2024-01-08")
]

new_df = spark.createDataFrame(new_data, columns)

df = df.union(new_df)

last_loaded_date = "2024-01-05"

incremental_df = df.filter(col("updated_date") > lit(last_loaded_date))

# Handle duplicates (keep latest record per order_id)
window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

incremental_df = incremental_df.withColumn(
    "rn", row_number().over(window_spec)
).filter(col("rn") == 1).drop("rn")

# Write incremental data
incremental_df.write.mode("append").parquet("/Volumes/workspace/default/outputs/load")

In [0]:
def run_pipeline(input_path, last_loaded_date):
    df = spark.read.parquet(input_path)

    df = df.withColumn("amount", col("amount").cast(IntegerType())) \
           .withColumn("updated_date", to_date(col("updated_date")))

    df = df.fillna({"amount": "1000"})

    df = df.withColumn("bonus", col("amount") * 0.1)

    df = df.withColumn(
        "category",
        when(col("amount") >= 20000, "High").otherwise("Low")
    )

    df = df.withColumn("amount_bucket", bucket_udf(col("amount")))

    # Incremental logic
    incremental_df = df.filter(col("updated_date") > lit(last_loaded_date))

    window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

    incremental_df = incremental_df.withColumn(
        "rn", row_number().over(window_spec)
    ).filter(col("rn") == 1).drop("rn")

    incremental_df.write.mode("append").parquet("/Volumes/workspace/default/outputs/load")